In [25]:
# Apriori
import pandas as pd, numpy as np
from itertools import combinations

df = pd.read_csv("/kaggle/input/datasets/roonil03alt/dmlab1/loan_approval.csv", dtype=str)
headers = df.columns.tolist()

cols = ["AgeGroup","IncomeLevel","CreditRating","Employed","Married","LoanApproved"]
T = [set(f"{c}={r[c]}" for c in cols) for _, r in df[cols].iterrows()]
min_support, min_confidence = 0.20, 0.70

def sup(s): return sum(s <= t for t in T) / len(T)

items = sorted({i for t in T for i in t})
L = [frozenset([i]) for i in items if sup({i}) >= min_support]
F, k = L[:], 2

while L:
    C = {a | b for i, a in enumerate(L) for b in L[i+1:] if len(a | b) == k}
    L = [c for c in C if sup(set(c)) >= min_support]
    F += L
    k += 1

R = []
for s in F:
    if len(s) < 2: continue
    for r in range(1, len(s)):
        for A in combinations(s, r):
            A = frozenset(A); B = s - A
            conf = sup(set(s)) / sup(set(A))
            if conf >= min_confidence:
                R.append((sorted(A), sorted(B), round(sup(set(s)),2), round(conf,2)))

print("Freq:", [sorted(x) for x in F if len(x) > 1][:3])
print("Rules:", R[:3])

Freq: [['CreditRating=Good', 'LoanApproved=Yes'], ['AgeGroup=Middle', 'LoanApproved=Yes'], ['AgeGroup=Senior', 'Married=No']]
Rules: [(['CreditRating=Good'], ['LoanApproved=Yes'], 0.21, 1.0), (['AgeGroup=Middle'], ['LoanApproved=Yes'], 0.23, 0.73), (['LoanApproved=Yes'], ['Employed=Yes'], 0.62, 0.91)]


In [22]:
# KMeans
import pandas as pd, numpy as np

df = pd.read_csv("/kaggle/input/datasets/roonil03alt/dmlab1/loan_approval.csv")
headers = df.columns.tolist()

X = df[["AgeNum","IncomeNum","CreditScoreNum","LoanAmountNum","DependentsNum"]].to_numpy(float)
k, method = 3, "euclidean"

def euclidean(a,b): return np.sqrt(((a-b)**2).sum())
def manhattan(a,b): return np.abs(a-b).sum()
def custom(a,b): return np.max(np.abs(a-b))   # Chebyshev

dist = {"euclidean": euclidean, "manhattan": manhattan, "custom": custom}[method]
C = X[:k].copy()

for _ in range(20):
    y = np.array([np.argmin([dist(x,c) for c in C]) for x in X])
    NC = np.array([X[y==i].mean(0) if (y==i).any() else C[i] for i in range(k)])
    if np.allclose(C, NC): break
    C = NC

print("Labels:", y.tolist())
print("Centroids:\n", np.round(C,2))

Labels: [1, 1, 1, 0, 2, 2, 2, 2, 2, 2, 0, 0, 1, 0, 1, 0, 2, 2, 2, 2, 0, 2, 0, 0, 2, 2, 1, 1, 0, 1, 1, 0, 2, 1, 0, 0, 0, 2, 2, 2, 0, 2, 2, 1, 0, 1, 1, 1]
Centroids:
 [[4.0530000e+01 8.6871930e+04 6.8107000e+02 1.7931470e+04 1.9300000e+00]
 [3.9290000e+01 1.1937293e+05 6.7107000e+02 4.3877930e+04 2.3600000e+00]
 [4.0680000e+01 4.9076680e+04 6.5447000e+02 4.2682050e+04 1.9500000e+00]]


In [2]:
# ID3 Decision Tree
import pandas as pd, numpy as np

df = pd.read_csv("/kaggle/input/datasets/roonil03alt/dmlab1/loan_approval.csv", dtype=str)
headers = df.columns.tolist()

X = ["AgeGroup","IncomeLevel","CreditRating","Employed","Married","Education","PropertyArea"]
Y = "LoanApproved"

def ent(y):
    p = y.value_counts(normalize=True).to_numpy()
    return -(p*np.log2(p)).sum()

def gain(d, f):
    return ent(d[Y]) - sum(len(s)/len(d)*ent(s[Y]) for _, s in d.groupby(f))

def id3(d, fs):
    if d[Y].nunique() == 1: return d[Y].iloc[0]
    if not fs: return d[Y].mode()[0]
    b = max(fs, key=lambda f: gain(d, f))
    return {b: {v: id3(s, [x for x in fs if x != b]) for v, s in d.groupby(b)}}

def pred(t, r):
    if not isinstance(t, dict): return t
    f = next(iter(t))
    return pred(t[f].get(r.get(f), df[Y].mode()[0]), r)

tree = id3(df, X)
p = np.array([pred(tree, r) for _, r in df[X].iterrows()])

print("Tree:", tree)
print("Acc:", (p == df[Y].to_numpy()).mean())

Tree: {'IncomeLevel': {'High': {'CreditRating': {'Excellent': 'Yes', 'Fair': 'Yes', 'Good': 'Yes', 'Poor': {'AgeGroup': {'Middle': 'Yes', 'Senior': 'Yes', 'Young': 'No'}}}}, 'Low': 'No', 'Medium': {'Education': {'Graduate': {'AgeGroup': {'Middle': 'Yes', 'Senior': {'CreditRating': {'Excellent': 'Yes', 'Fair': 'No', 'Good': 'Yes', 'Poor': 'Yes'}}, 'Young': 'Yes'}}, 'PostGrad': 'Yes', 'School': {'CreditRating': {'Excellent': 'No', 'Fair': 'Yes', 'Good': 'Yes', 'Poor': 'No'}}}}}}
Acc: 1.0


In [13]:
# Multinomial NB
import pandas as pd, numpy as np

df = pd.read_csv("/kaggle/input/datasets/roonil03alt/dmlab1/loan_approval.csv", dtype=str)
headers = df.columns.tolist()

X = pd.get_dummies(df[["AgeGroup","IncomeLevel","CreditRating","Employed","Married","Education","PropertyArea"]]).astype(int).to_numpy()
y = df["LoanApproved"].to_numpy()
cls, alpha = np.unique(y), 1.0

pr = {c: (y == c).mean() for c in cls}
pb = {}
for c in cls:
    cnt = X[y == c].sum(0) + alpha
    pb[c] = cnt / cnt.sum()

def pred1(x):
    s = [np.log(pr[c]) + (x*np.log(pb[c])).sum() for c in cls]
    return cls[np.argmax(s)]

yp = np.array([pred1(x) for x in X])

def metrics(a, b):
    labs = np.unique(np.r_[a, b]) 
    cm = pd.DataFrame(0, index=labs, columns=labs)
    for x, y in zip(a, b): cm.loc[x, y] += 1
    acc = (a == b).mean()
    prec = np.mean([cm.loc[c,c] / max(cm[c].sum(), 1) for c in labs])
    rec  = np.mean([cm.loc[c,c] / max(cm.loc[c].sum(), 1) for c in labs])
    return cm, acc, prec, rec

cm, acc, prec, rec = metrics(y, yp)
print("CM:\n", cm)
print("Acc:", round(acc,3), "Prec:", round(prec,3), "Rec:", round(rec,3))

CM:
      No  Yes
No   13    2
Yes   1   32
Acc: 0.938 Prec: 0.935 Rec: 0.918


In [20]:
# Gaussian NB
import pandas as pd, numpy as np

df = pd.read_csv("/kaggle/input/datasets/roonil03alt/dmlab1/loan_approval.csv")
headers = df.columns.tolist()

X = df[["AgeNum","IncomeNum","CreditScoreNum","LoanAmountNum","DependentsNum"]].to_numpy(float)
y = df["LoanApproved"].to_numpy()
cls = np.unique(y)

pr = {c: (y == c).mean() for c in cls}
mu = {c: X[y == c].mean(0) for c in cls}
va = {c: X[y == c].var(0) + 1e-9 for c in cls}

def pred1(x):
    s = [np.log(pr[c]) - .5*np.sum(np.log(2*np.pi*va[c]) + ((x-mu[c])**2)/va[c]) for c in cls]
    return cls[np.argmax(s)]

yp = np.array([pred1(x) for x in X])

def metrics(a, b):
    labs = np.unique(np.r_[a, b])
    cm = pd.DataFrame(0, index=labs, columns=labs)
    for x, y in zip(a, b): cm.loc[x, y] += 1
    acc = (a == b).mean()
    prec = np.mean([cm.loc[c,c] / max(cm[c].sum(), 1) for c in labs])
    rec  = np.mean([cm.loc[c,c] / max(cm.loc[c].sum(), 1) for c in labs])
    return cm, acc, prec, rec

cm, acc, prec, rec = metrics(y, yp)
print("CM:\n", cm)
print("Acc:", round(acc,3), "Prec:", round(prec,3), "Rec:", round(rec,3))

CM:
      No  Yes
No   11    4
Yes   2   31
Acc: 0.875 Prec: 0.866 Rec: 0.836


In [1]:
# Interactive ID3
import pandas as pd, numpy as np

df = pd.read_csv("/kaggle/input/datasets/roonil03alt/dmlab1/loan_approval.csv", dtype=str)
headers = df.columns.tolist()

X = ["AgeGroup","IncomeLevel","CreditRating","Employed","Married","Education","PropertyArea"]
Y = "LoanApproved"

def ent(y):
    p = y.value_counts(normalize=True).to_numpy()
    return -(p*np.log2(p)).sum()

def gain(d, f):
    return ent(d[Y]) - sum(len(s)/len(d)*ent(s[Y]) for _, s in d.groupby(f))

def id3(d, fs):
    if d[Y].nunique() == 1: return d[Y].iloc[0]
    if not fs: return d[Y].mode()[0]
    b = max(fs, key=lambda f: gain(d, f))
    return {b: {v: id3(s, [x for x in fs if x != b]) for v, s in d.groupby(b)}}

def pred(t, r):
    if not isinstance(t, dict): return t
    f = next(iter(t))
    return pred(t[f].get(r.get(f), df[Y].mode()[0]), r)

tree = id3(df, X)
row = {f: input(f + ": ") for f in X}
print("Pred:", pred(tree, row))

KeyboardInterrupt: Interrupted by user